# N-BEATS Forecast Demo (synthetic AR data)

A minimal end-to-end demo of the `tfts` N-BEATS model: generate the synthetic AR series, build target-only windows (`x` = lookback, `y` = forecast), train a small interpretable N-BEATS (trend + seasonality stacks), then plot a multi-step forecast.

Model config (defaults of `AutoConfig.for_model("nbeats")`): widths `[32,512]`, num_blocks `[3,3]`, num_block_layers `[3,3]`, expansion_coefficient_lengths `[3,7]`, dropout `0.1`.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("../.."))  # make the local ./tfts package importable

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

tf.random.set_seed(42)
np.random.seed(42)

ENCODER_LENGTH = 60  # lookback (context) window
PREDICTION_LENGTH = 20  # forecast horizon
BATCH_SIZE = 128

## 1. Data
Generate the synthetic AR data (quadratic trend + seasonality + noise, 100 series x 400 steps) using the same `generate_ar_data(...)` call and seed as the tutorial.

In [ ]:
from tfts.data.ar import ARNBeatsPreprocessor, generate_ar_data

data = generate_ar_data(seasonality=10.0, timesteps=400, n_series=100, seed=42)
print(data.head())
print(
    "rows:",
    len(data),
    "| series:",
    data.series.nunique(),
    "| time range:",
    data.time_idx.min(),
    "-",
    data.time_idx.max(),
)

## 2. Windowing (target-only)
N-BEATS is univariate, so `x` is just the lookback window of `value` and `y` the following forecast window. No covariates, no normalization.

In [ ]:
proc = ARNBeatsPreprocessor(data, encoder_length=ENCODER_LENGTH, prediction_length=PREDICTION_LENGTH)
train_batch = proc.train()  # sliding windows fully in time_idx <= training_cutoff
val_batch = proc.validation()  # one forecast per series starting at training_cutoff+1
print("train x/y:", train_batch.x.shape, train_batch.y.shape)
print("val   x/y:", val_batch.x.shape, " ", val_batch.y.shape)

train_ds = tf.data.Dataset.from_tensor_slices((train_batch.x, train_batch.y)).shuffle(5000, seed=42).batch(BATCH_SIZE)
val_ds = tf.data.Dataset.from_tensor_slices((val_batch.x, val_batch.y)).batch(BATCH_SIZE)

## 3. Build the model
Use the `tfts` AutoModel registry (the N-BEATS implementation lives in `tfts/models/nbeats.py`).

In [ ]:
from tfts.models.auto_config import AutoConfig
from tfts.models.auto_model import AutoModel

cfg = AutoConfig.for_model("nbeats")  # interpretable default config
print("widths:", cfg.widths, "| num_blocks:", cfg.num_blocks, "| num_block_layers:", cfg.num_block_layers)

model = AutoModel.from_config(cfg, predict_sequence_length=PREDICTION_LENGTH)
km = model.build_model(tf.keras.Input(shape=(ENCODER_LENGTH, 1)))  # Keras functional model
km.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-2), loss="mae", metrics=["mae"])
km.summary(line_length=90)

## 4. Train
A short demo budget (a few epochs). Loss is MAE, the point-forecast equivalent of MASE.

In [ ]:
history = km.fit(train_ds, validation_data=val_ds, epochs=8, verbose=1)

plt.figure(figsize=(9, 3.5))
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.xlabel("epoch")
plt.ylabel("MAE")
plt.legend()
plt.title("Training / validation MAE")
plt.tight_layout()
plt.show()

## 5. Evaluate & plot a forecast

In [ ]:
preds = km.predict(val_ds, verbose=0)[..., 0]  # (n_series, PREDICTION_LENGTH)
actual = val_batch.y[..., 0]
err = actual - preds
print("MAE:", float(np.abs(err).mean()))
print("MSE:", float(np.mean(err**2)))

In [ ]:
s = 0  # pick a series
window = val_batch.x[s, :, 0]  # history (encoder window)
fut = np.arange(ENCODER_LENGTH, ENCODER_LENGTH + PREDICTION_LENGTH)

plt.figure(figsize=(10, 4))
plt.plot(np.arange(-ENCODER_LENGTH, 0), window, label="history")
plt.plot(fut, actual[s], label="actual", marker="o")
plt.plot(fut, preds[s], label="forecast", marker="x")
plt.axvline(0, color="gray", ls="--")
plt.xlabel("time (relative to forecast start)")
plt.ylabel("value")
plt.legend()
plt.title(f"Series {s}: {PREDICTION_LENGTH}-step N-BEATS forecast")
plt.tight_layout()
plt.show()